This is the Quality control step of crc-pm

In [1]:
# modules
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy as sp
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
from pathlib import Path
import anndata as ad
import sys
import re
import os
import seaborn as sns
import scvi
### Import personal Utensils
sys.path.insert(0, "..")
import utils as ut

In [2]:
# read data
CuratedDir = Path("../../result/curated/merged.h5ad")
adata = sc.read_h5ad(CuratedDir)
adata.X = adata.layers['counts'].copy()

### Remove ambient RNA

In [ ]:
# gentle QC
sc.pp.filter_cells(adata, min_counts=400)
sc.pp.filter_cells(adata, min_genes=100)

# scAR removal of ambient RNA
denoised_rna = adata.layers["counts"].copy().tolil()
for study in adata.obs["Study"].unique():
    mask = adata.obs["Study"] == study
    idx = np.flatnonzero(mask.to_numpy())
    sub = adata[idx].copy()
    scvi.external.SCAR.setup_anndata(sub,batch_key="Glob_Sample")
    model = scvi.external.SCAR(sub,ambient_profile=None)
    model.train(accelerator="gpu")
    denoised = model.get_denoised_counts()
    # write results into the corresponding rows of the full matrix
    denoised_rna[idx, :] = csr_matrix(denoised)
    print(f"{study} done: {denoised.shape}")
# save back to the original AnnData and to .X
adata.layers["denoised_rna"] = denoised_rna.tocsr()
adata.X = adata.layers["denoised_rna"].copy()

### Coarse QC

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var['gene_name'].str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var['gene_name'].str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var['gene_name'].str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True)

In [ ]:
# Preview data quality by sample
ut.preview_cell_quality(adata, "Study",'crc10-SMC')

In [ ]:
p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
p2 = sc.pl.violin(adata, "pct_counts_mt")
p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
p4 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="Study")

In [ ]:
def is_outlier(adata, metric: str, nmads: int, groupby):
    M = adata.obs[metric]
    grouped = adata.obs.groupby(groupby,observed=True)[metric]
    median = grouped.transform("median")
    mad = grouped.transform(lambda x: median_abs_deviation(x, nan_policy="omit"))
    outlier = ((M < median - nmads * mad) | (M > median + nmads * mad))
    return outlier

In [ ]:
# cell filtering
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", 5, groupby="Study")
    | is_outlier(adata, "log1p_n_genes_by_counts", 5, groupby="Study")
    | is_outlier(adata, "pct_counts_in_top_20_genes", 5, groupby="Study")
)
adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 3, groupby="Study") | (
    adata.obs["pct_counts_mt"] > 40)
# very hard to choose the mt threshold, as lots of cells in crc01 have high mt percentage
# while in crc06, almost no mt gene
# it is so decided that 40% should be should be the threshold

adata.obs["qc_outlier"] = adata.obs["outlier"] | adata.obs["mt_outlier"]

qc_summary = (
    adata.obs
    .groupby("Study", observed=True)["outlier"]
    .agg(total="size", removed="sum")
)
qc_summary["kept"] = qc_summary["total"] - qc_summary["removed"]
qc_summary["removed_pct"] = qc_summary["removed"] / qc_summary["total"] * 100

print(qc_summary)

In [ ]:
# qc
adata = adata[~adata.obs["qc_outlier"]].copy()
# gene filtering
sc.pp.filter_genes(adata, min_cells=20)

### Doublet depletion

In [ ]:
adata.obs["Study_Tissue"] = adata.obs["Study"].astype(str) + "_" + adata.obs["Tissue"].astype(str)
sc.pp.scrublet(
    adata,
    batch_key="Study_Tissue",
    expected_doublet_rate=0.05,
    random_state=0,
)
# here we use study_tissue as batch_key, because fucking crc01 contains some sample which literally has no fucking cells

In [ ]:
sc.pl.scrublet_score_distribution(adata)

### Export

In [ ]:
adata = adata[~adata.obs["predicted_doublet"]].copy()
outdir = Path("../../result/qc")
adata.write_h5ad(outdir / "qc.h5ad", compression="lzf")